## import thư viện

In [1]:
!pip install transformers==4.57.3 accelerate==1.12.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import json

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from datasets import load_dataset, Dataset, concatenate_datasets

2026-06-23 09:17:50.652679: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782206270.865411      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782206270.926031      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782206271.427176      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782206271.427218      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782206271.427221      23 computation_placer.cc:177] computation placer alr

In [3]:
# Kiểm tra GPU
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


## import dataset ViNLI + large (250k)

In [4]:
large_dataset = pd.read_csv("/kaggle/input/datasets/huynhdat2010/rerank-finetune-data/ViNLI_and_large.csv")
large_dataset = Dataset.from_pandas(large_dataset)

# Shuffle dữ liệu
large_dataset = large_dataset.shuffle(seed=42)

# Kiểm tra thông tin
print(f"Size: {large_dataset.shape}")

Size: (505956, 3)


## chia train/dev/test

In [5]:
split = large_dataset.train_test_split(test_size=0.003, seed=42)
temp_train = split["train"]
test_dataset_large_dataset = split["test"]

train_valid = temp_train.train_test_split(test_size=0.003, seed=42)
train_dataset_large_dataset = train_valid["train"]
valid_dataset_large_dataset = train_valid["test"]

In [6]:
print("Train:", len(train_dataset_large_dataset))
print("Valid:", len(valid_dataset_large_dataset))
print("Test:", len(test_dataset_large_dataset))

Train: 502924
Valid: 1514
Test: 1518


## Load model


In [7]:
models = [
    "cross-encoder/ms-marco-MiniLM-L6-v2",
    "Data-Lab/bge-reranker-v2-m3-cross-encoder-v0.1",
    "jinaai/jina-reranker-v2-base-multilingual",
    "Alibaba-NLP/gte-multilingual-reranker-base",
    "BAAI/bge-reranker-base",
    "AITeamVN/Vietnamese_Reranker",
    "huynhdat543/VietNamese_law_rerank_v1",
    "huynhdat543/VietNamese_law_rerank_v2",
    "huynhdat543/VietNamese_law_rerank_v3",
    "huynhdat543/VietNamese_law_rerank_v4",
    "huynhdat543/VietNamese_law_rerank_v5"
]

In [8]:
model_store = {}

print("Loading model...\n")

for name in models:
    try:
        print(f"Loading {name} ...")
        tokenizer = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
        model = AutoModelForSequenceClassification.from_pretrained(name, trust_remote_code=True).to(device)
        model.eval()

        model_store[name] = {
            "tokenizer": tokenizer,
            "model": model
        }
        print(f"Loaded {name}\n")

    except Exception as e:
        print(f"Lỗi khi load model {name}: {e}")

print("Tải xong tất cả model\n")

Loading model...

Loading cross-encoder/ms-marco-MiniLM-L6-v2 ...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/huynhdat543/VietNamese_law_rerank_v5:
- configuration.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/huynhdat543/VietNamese_law_rerank_v5:
- modeling.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Loaded huynhdat543/VietNamese_law_rerank_v5

Tải xong tất cả model



## Evaluation Large Dataset

In [9]:
def batched_predict(model, tokenizer, pairs, batch_size=1):
    model.eval()
    scores = []

    with torch.no_grad():
        for i in range(0, len(pairs), batch_size):
            batch = pairs[i:i + batch_size]
            queries = [q for q, d in batch]
            docs = [d for q, d in batch]

            encoded = tokenizer(
                queries,
                docs,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            ).to(model.device)

            logits = model(**encoded).logits

            if logits.shape[-1] == 1:
                batch_scores = torch.sigmoid(logits).squeeze().tolist()
            else:
                batch_scores = torch.softmax(logits, dim=1)[:, 1].tolist()

            scores.extend(batch_scores)

            torch.cuda.empty_cache()

    return scores

In [10]:
def evaluate_loaded_model(model_name, model_dict, test_dataset, batch_size=1):
    print(f"\n--- Evaluating model: {model_name} ---")

    model = model_dict["model"]
    tokenizer = model_dict["tokenizer"]

    test_pairs = list(zip(test_dataset["query"], test_dataset["document"]))
    true_labels = test_dataset["label"]

    pred_scores = batched_predict(model, tokenizer, test_pairs, batch_size=batch_size)
    pred_labels = [1 if s >= 0.5 else 0 for s in pred_scores]

    acc = accuracy_score(true_labels, pred_labels)
    f1 = f1_score(true_labels, pred_labels)
    precision = precision_score(true_labels, pred_labels)
    recall = recall_score(true_labels, pred_labels)

    print("Classification Evaluation:")
    print(f"Accuracy:  {acc:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")

    return {
        "model": model_name,
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
    }


In [11]:
results = []

for name, model_dict in model_store.items():
    try:
        res = evaluate_loaded_model(name, model_dict, test_dataset_large_dataset, batch_size=8)
        results.append(res)
    except Exception as e:
        print(f"Lỗi khi evaluate model {name}: {e}")

print("\n--- Test on Large_Dataset ---")
pd.DataFrame(results)


--- Evaluating model: cross-encoder/ms-marco-MiniLM-L6-v2 ---
Classification Evaluation:
Accuracy:  0.5356
F1 Score:  0.6750
Precision: 0.5148
Recall:    0.9799

--- Evaluating model: Data-Lab/bge-reranker-v2-m3-cross-encoder-v0.1 ---
Classification Evaluation:
Accuracy:  0.5711
F1 Score:  0.6959
Precision: 0.5344
Recall:    0.9973

--- Evaluating model: jinaai/jina-reranker-v2-base-multilingual ---
Classification Evaluation:
Accuracy:  0.6232
F1 Score:  0.7008
Precision: 0.5751
Recall:    0.8969

--- Evaluating model: Alibaba-NLP/gte-multilingual-reranker-base ---
Classification Evaluation:
Accuracy:  0.5929
F1 Score:  0.7026
Precision: 0.5485
Recall:    0.9772

--- Evaluating model: BAAI/bge-reranker-base ---
Classification Evaluation:
Accuracy:  0.6423
F1 Score:  0.6625
Precision: 0.6183
Recall:    0.7135

--- Evaluating model: AITeamVN/Vietnamese_Reranker ---
Classification Evaluation:
Accuracy:  0.7036
F1 Score:  0.6853
Precision: 0.7174
Recall:    0.6560

--- Evaluating model: h

,model,accuracy,f1,precision,recall
0,cross-encoder/ms-marco-MiniLM-L6-v2,0.535573,0.674965,0.514768,0.979920
1,Data-Lab/bge-reranker-v2-m3-cross-encoder-v0.1,0.571146,0.695936,0.534433,0.997323
2,jinaai/jina-reranker-v2-base-multilingual,0.623188,0.700837,0.575107,0.896921
3,Alibaba-NLP/gte-multilingual-reranker-base,0.592885,0.702599,0.548460,0.977242
4,BAAI/bge-reranker-base,0.642292,0.662523,0.618329,0.713521
5,AITeamVN/Vietnamese_Reranker,0.703557,0.685315,0.717423,0.655957
6,huynhdat543/VietNamese_law_rerank_v1,0.695652,0.755037,0.625110,0.953146
7,huynhdat543/VietNamese_law_rerank_v2,0.605402,0.710488,0.555976,0.983936
8,huynhdat543/VietNamese_law_rerank_v3,0.847826,0.865149,0.767081,0.991968
9,huynhdat543/VietNamese_law_rerank_v4,0.507905,0.666667,0.500000,1.000000
